# Exercice 5 — Comité modèles : sur-apprentissage, seuil de décision et classement des cas d'usage

## Contexte métier
Vous participez au comité modèles. Trois sujets vous attendent : (1) un data scientist a choisi la complexité d'un modèle de **rachat par ancienneté** — a-t-il évité le sur-apprentissage, et avec quel protocole ? (2) la direction veut déclencher une **campagne de fidélisation** : à partir de quelle probabilité, sachant qu'une campagne inutile coûte 80 € et un rachat non anticipé 1 500 € ? (3) dix cas d'usage IA sont sur la table : quelle famille de modèles, quel risque principal ?

## Objectifs
- Distinguer **sous-ajustement, bon compromis et sur-apprentissage** via les erreurs train / validation / test
- Respecter le protocole **train → validation → test (une seule fois)**
- Choisir un **seuil de décision** par le coût métier à partir d'une matrice de confusion, et lire une AUC
- Classer des cas d'usage vie par **famille de modèles** et associer à chaque situation son **risque principal**

## Rappels théoriques mobilisés (slides de la session)
- *Généralisation et sur-apprentissage : l'image à retenir* — sous-ajusté / bon compromis / sur-ajusté
- *Le compromis biais–variance* et *Risque empirique, risque réel* — train / validation / test
- *Le formalisme général de l'apprentissage supervisé* — le seuil optimal minimise un coût, pas une distance
- *Lire une performance : matrice de confusion et courbe ROC*
- *Quatre familles de modèles* et *Typologie des cas d'usage en actuariat vie*
- *Risques transverses* — fuite, biais d'échantillon, sur-évaluation, dérive, opacité, automatisation excessive

## Mini-dictionnaire & prérequis
| Élément | Signification / usage |
|---|---|
| Degré du polynôme | complexité du modèle : plus il est élevé, plus la courbe est flexible |
| `np.polyfit(z, y, d)` / `np.polyval(coefs, z)` | ajuste / évalue un polynôme de degré `d` par moindres carrés |
| MSE train / validation / test | erreur sur les données d'apprentissage / de réglage / d'évaluation finale |
| Matrice de confusion | VP, FP, FN, VN : bons rachats détectés, fausses alertes, rachats manqués, vrais négatifs |
| Coût d'un seuil `s` | c_FP × FP(s) + c_FN × FN(s) |
| Seuil théorique | pour des probabilités calibrées : s* = c_FP / (c_FP + c_FN) |
| Familles | `supervisé`, `non supervisé`, `renforcement`, `génératif`, `LLM/RAG` |
| `_h("…")` | outil fourni pour vérifier vos réponses sans les afficher |

## Mode d'emploi
- Complétez les cellules marquées `...  # À COMPLÉTER` puis exécutez-les **dans l'ordre**.
- Les cellules **Vérification** vous disent immédiatement si vous êtes sur la bonne piste (✅ / ❌ + indice).
- Bloqué ? Ouvrez les **💡 indices** progressifs avant de demander la correction.
- À la fin, la cellule `bilan()` donne votre score.

In [ ]:
# Outil d'auto-vérification — NE PAS MODIFIER
import numpy as np, pandas as pd, matplotlib.pyplot as plt, hashlib
_resultats = {}
def verifier(nom, condition, indice=""):
    """Évalue `condition` (une fonction sans argument) ; affiche ✅ ou ❌ + indice."""
    try: ok = bool(condition())
    except Exception: ok = False
    _resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok or not indice else "\n    💡 " + indice))
def bilan():
    n, k = len(_resultats), sum(_resultats.values())
    print(f"\n🏁 Score : {k}/{n} vérifications réussies")
    print("   Bravo, exercice terminé !" if k == n and n > 0 else "   Reprenez les ❌ ci-dessus (relancez la cellule concernée puis `bilan()`).")
def _h(s): return hashlib.sha1(s.encode()).hexdigest()[:8]
%matplotlib inline

---
## Partie A — Sur-apprentissage et protocole train / validation / test

On observe le taux de rachat (%) en fonction de l'**ancienneté** du contrat (courbe en cloche autour de l'échéance fiscale, plus une dérive), sur 120 observations bruitées. On compare des modèles polynomiaux de degré 1 à 12.

In [ ]:
rng = np.random.default_rng(21)
n = 120
anciennete = rng.uniform(0.5, 15, n)
vraie = 4 + 6 * np.exp(-((anciennete - 6) ** 2) / (2 * 2.5 ** 2)) + 0.25 * anciennete
y = vraie + rng.normal(0, 1.2, n)
idx = rng.permutation(n)
i_tr, i_va, i_te = idx[:25], idx[25:60], idx[60:]          # 25 train, 35 validation, 60 test
mu, sd = anciennete[i_tr].mean(), anciennete[i_tr].std()   # μ, σ estimés sur le TRAIN seul
z = (anciennete - mu) / sd
z_tr, y_tr, z_va, y_va, z_te, y_te = z[i_tr], y[i_tr], z[i_va], y[i_va], z[i_te], y[i_te]
plt.scatter(z_tr, y_tr, label="train"); plt.scatter(z_va, y_va, label="validation", alpha=.6); plt.legend()
plt.xlabel("ancienneté standardisée"); plt.ylabel("taux de rachat (%)"); plt.show()

**Q1.** Pour chaque degré `d` de 1 à 12 : ajustez le polynôme **sur le train**, et stockez `mse_train[d]` et `mse_val[d]`. Choisissez `d_opt` (minimum de l'erreur de **validation**) puis calculez **une seule fois** `mse_test` pour ce degré.

<details><summary>💡 Indice 1</summary>

`coefs = np.polyfit(z_tr, y_tr, d)` ; prédictions : `np.polyval(coefs, z_tr)`.

</details>

<details><summary>💡 Indice 2</summary>

`mse_train[d] = np.mean((np.polyval(coefs, z_tr) - y_tr) ** 2)` ; idem avec `z_va`, `y_va`.

</details>

<details><summary>💡 Indice 3</summary>

`d_opt = min(mse_val, key=mse_val.get)`

</details>

In [ ]:
mse_train, mse_val = {}, {}
for d in range(1, 13):
    coefs = ...              # À COMPLÉTER
    mse_train[d] = ...       # À COMPLÉTER
    mse_val[d] = ...         # À COMPLÉTER
d_opt = ...                  # À COMPLÉTER
mse_test = ...               # À COMPLÉTER : erreur test du degré d_opt, calculée UNE fois
plt.plot(list(mse_train), list(mse_train.values()), "o-", label="train")
plt.plot(list(mse_val), list(mse_val.values()), "s-", label="validation")
plt.yscale("log"); plt.axvline(d_opt, ls="--", c="gray"); plt.xlabel("degré"); plt.ylabel("MSE (log)"); plt.legend(); plt.show()
print(f"Degré retenu : {d_opt} | MSE test : {mse_test:.2f}")

In [ ]:
verifier("Q1 · 12 degrés évalués", lambda: len(mse_train) == 12 and len(mse_val) == 12 and all(np.isfinite(v) for v in list(mse_train.values()) + list(mse_val.values())))
verifier("Q1 · l'erreur train baisse quand le degré augmente", lambda: mse_train[12] < mse_train[6] < mse_train[1])
verifier("Q1 · sur-apprentissage visible (validation du degré 12 ≫ optimum)", lambda: mse_val[12] > 10 * min(mse_val.values()))
verifier("Q1 · degré retenu", lambda: d_opt == 6, "minimum de la courbe de VALIDATION")
verifier("Q1 · MSE test", lambda: np.isclose(mse_test, 1.904832, atol=0.05))

**Q1c · QCM.** Pourquoi choisit-on le degré sur la **validation** et non sur le **test** ?
- **A.** Parce que le test est trop petit
- **B.** Parce que le test sert à estimer le risque réel *une seule fois* ; s'en servir pour choisir en ferait une validation, et l'estimation finale ne serait plus honnête
- **C.** Parce que le test n'a pas de labels

De plus, que vaut l'erreur **train** du degré 12 ? (`reponse_train12` = `"bonne mesure de la performance"` ou `"trompeuse"`).

In [ ]:
reponse_q1c = ...       # "A", "B" ou "C"
reponse_train12 = ...   # "bonne mesure de la performance" ou "trompeuse"

In [ ]:
verifier("Q1c · protocole", lambda: reponse_q1c == "B")
verifier("Q1c · l'erreur train est trompeuse", lambda: reponse_train12 == "trompeuse", "elle mesure la mémorisation, pas la généralisation")

## Partie B — Choisir un seuil par le coût, pas par habitude

Un score de rachat **calibré** `p` est disponible pour 4 000 contrats. Une campagne coûte **80 €** par contrat ciblé à tort (faux positif), un rachat non anticipé **1 500 €** (faux négatif).

In [ ]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(5)
N = 4000
p = rng.beta(2, 18, N)             # score calibré par construction
y_r = rng.binomial(1, p)           # rachat observé
c_fp, c_fn = 80, 1500
print(f"Taux de rachat : {y_r.mean():.1%}")

**Q2.** (a) Écrivez `confusion(p, y, s)` qui renvoie `(VN, FP, FN, VP)` pour la règle « cibler si p ≥ s ». (b) Calculez le coût total pour chaque seuil de `grille` (0,01 → 0,59) → `couts` (tableau) ; `s_opt` = seuil de coût minimal. (c) `cout_05` = coût au seuil 0,5 et `s_theo = c_fp/(c_fp+c_fn)`. (d) `auc` du score.

<details><summary>💡 Indice 1</summary>

`pred = p >= s` ; FP : `(pred & (y == 0)).sum()` ; FN : `((~pred) & (y == 1)).sum()`.

</details>

<details><summary>💡 Indice 2</summary>

`couts = np.array([c_fp * confusion(p, y_r, s)[1] + c_fn * confusion(p, y_r, s)[2] for s in grille])`

</details>

<details><summary>💡 Indice 3</summary>

`s_opt = grille[np.argmin(couts)]`

</details>

In [ ]:
def confusion(p, y, s):
    pred = p >= s
    ...   # À COMPLÉTER : renvoie (VN, FP, FN, VP)

grille = np.round(np.arange(0.01, 0.60, 0.01), 2)
couts = ...      # À COMPLÉTER : coût total pour chaque seuil de la grille
s_opt = ...      # À COMPLÉTER
cout_05 = ...    # À COMPLÉTER : coût au seuil 0,5
s_theo = ...     # À COMPLÉTER
auc = ...        # À COMPLÉTER
plt.plot(grille, couts); plt.axvline(s_opt, c="g", ls="--", label=f"s_opt={s_opt}"); plt.axvline(0.5, c="r", ls=":", label="0,5")
plt.xlabel("seuil s"); plt.ylabel("coût total (€)"); plt.legend(); plt.show()
print(f"s_opt={s_opt} | s_théorique={s_theo:.3f} | coût(s_opt)={couts.min():,.0f} € | coût(0,5)={cout_05:,.0f} € | AUC={auc:.3f}")

In [ ]:
verifier("Q2a · confusion au seuil 0,2", lambda: tuple(confusion(p, y_r, 0.2)) == (3351, 251, 312, 86))
verifier("Q2a · les 4 cases totalisent N", lambda: sum(confusion(p, y_r, 0.3)) == N)
verifier("Q2b · seuil optimal très inférieur à 0,5", lambda: 0.01 <= s_opt <= 0.12, "un faux négatif coûte ≈ 19 fois un faux positif")
verifier("Q2c · coût à 0,5 supérieur d'au moins 50 % au coût optimal", lambda: cout_05 > 1.5 * couts.min())
verifier("Q2c · seuil théorique", lambda: np.isclose(s_theo, 80 / 1580))
verifier("Q2d · AUC", lambda: np.isclose(auc, 0.712935, atol=1e-3))

**Q2e · QCM.** Le seuil par défaut de 0,5 est-il adapté ici ? **A.** Oui, c'est la valeur standard · **B.** Non : le coût d'un faux négatif étant très supérieur à celui d'un faux positif, le seuil optimal est bien plus bas · **C.** Non, il faut le fixer à 0,9. (`reponse_q2e`)

In [ ]:
reponse_q2e = ...   # "A", "B" ou "C"

In [ ]:
verifier("Q2e · seuil et coûts asymétriques", lambda: reponse_q2e == "B")

## Partie C — Classer des cas d'usage : famille de modèles

Les dix cas de l'atelier. Pour chacun, indiquez la **famille dominante** dans le dictionnaire `familles` parmi : `"supervisé"`, `"non supervisé"`, `"renforcement"`, `"génératif"`, `"LLM/RAG"`.

| # | Cas d'usage |
|---|---|
| 1 | Accélérer le calcul du Best Estimate par modèle proxy |
| 2 | Segmenter le portefeuille pour la campagne de fidélisation |
| 3 | Projeter la courbe des taux sur 30 ans |
| 4 | Détecter les rachats atypiques en temps quasi réel |
| 5 | Résumer les notes actuarielles pour le comité modèles |
| 6 | Prédire les versements complémentaires à 12 mois |
| 7 | Générer des scénarios de stress économiques rares |
| 8 | Assistant de recherche dans la documentation Solvabilité II |
| 9 | Optimiser l'allocation d'actifs par essais simulés |
| 10 | Extraire les clauses clés de contrats scannés |

<details><summary>💡 Indice 1</summary>

Critère discriminant : **quel signal** reçoit le modèle ? une cible connue (supervisé), aucune cible (non supervisé), une récompense séquentielle (renforcement), la distribution elle-même (génératif), ou du texte/documents (LLM/RAG).

</details>

<details><summary>💡 Indice 2</summary>

Cas 3 : deux réponses défendables (trajectoire prédite à partir de l'historique, ou scénarios générés).

</details>

In [ ]:
familles = {
    1: ..., 2: ..., 3: ..., 4: ..., 5: ...,
    6: ..., 7: ..., 8: ..., 9: ..., 10: ...,
}   # À COMPLÉTER

In [ ]:
_acc = {1: ['7e53fee8'], 2: ['0014cab9'], 3: ['cb21fb76', '1c90961b'], 4: ['e25a428c'], 5: ['dc05ee77'], 6: ['2f52b1c2'], 7: ['1f35da30'], 8: ['3a38c7ab'], 9: ['829b6caa'], 10: ['386580f8']}
for _i in range(1, 11):
    verifier(f"Q3 · cas {_i}", lambda: _h(f"{_i}:{familles[_i]}") in _acc[_i], "quel signal d'apprentissage est disponible ?")

## Partie D — Quel risque principal ?

Associez à chaque situation son **risque principal** parmi : `"fuite"`, `"biais d'échantillon"`, `"sur-évaluation"`, `"dérive"`, `"opacité"`, `"automatisation excessive"` (chaque risque est utilisé une fois).

- **a.** Le score de rachat utilise comme variable le montant effectivement racheté en N+1.
- **b.** Le modèle est entraîné sur le canal courtage mais déployé sur le réseau d'agents.
- **c.** Le jeu de test a été réutilisé une quinzaine de fois pour régler les hyperparamètres.
- **d.** Après la hausse des taux, les rachats doublent et le modèle les sous-estime.
- **e.** Personne ne sait expliquer au validateur pourquoi un contrat obtient 0,4.
- **f.** Les offres de fidélisation sont envoyées automatiquement, sans relecture humaine, à tous les contrats scorés.

In [ ]:
risques = {"a": ..., "b": ..., "c": ..., "d": ..., "e": ..., "f": ...}   # À COMPLÉTER

In [ ]:
_att = {'a': '6c6ec040', 'b': '68f46d63', 'c': '77c4a478', 'd': 'c3c6c7c8', 'e': 'b7636123', 'f': 'e988fe90'}
for _k in "abcdef":
    verifier(f"Q4 · situation {_k}", lambda: _h(f"{_k}:{risques[_k]}") == _att[_k], "relisez la slide « Risques transverses »")

In [ ]:
bilan()